# 03. QLoRA Instruction Tuning

**Topics covered:** 4-bit Quantization · QLoRA · Instruction Tuning

This notebook builds on [02_lora_finetuning.ipynb](https://github.com/S33mi/modern-ai-llm-journey/blob/main/03_finetuning/02_lora_finetuning.ipynb). We combine **4-bit quantization** with LoRA so that even larger models can be fine-tuned on a single consumer GPU.

We will:
1. Understand **4-bit quantization** (NF4) and why it saves memory
2. Configure **QLoRA** with `bitsandbytes` + PEFT
3. Prepare an **instruction-following** dataset
4. Fine-tune a small causal LM with the instruction format
5. Generate responses and save the adapter

## 1. Setup & Imports

```bash
pip install transformers datasets accelerate peft bitsandbytes trl
```

> **Note:** `bitsandbytes` requires a CUDA GPU for 4-bit loading. On CPU-only machines the notebook will still explain the concepts; switch to an 8-bit or full-precision baseline if needed.

In [5]:
pip install transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.1 MB/s eta 0:00:00


In [6]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cpu
CUDA available: False


## 2. 4-bit Quantization & QLoRA Theory

### The memory problem

A 7 B parameter model in float16 already needs ~14 GB just for the weights.  
Optimizer states (Adam) roughly *triple* that during full fine-tuning.

### Quantization

Store weights in fewer bits:

| Format | Bits | Approx. size (7 B) |
|--------|------|--------------------|
| FP32 | 32 | ~28 GB |
| FP16 / BF16 | 16 | ~14 GB |
| INT8 | 8 | ~7 GB |
| **NF4 (4-bit)** | 4 | **~3.5 GB** |

**NormalFloat4 (NF4)** is an information-theoretically optimal 4-bit quantisation for normally distributed weights (introduced in the QLoRA paper).

### QLoRA = Quantized base + LoRA adapters

1. Load the base model in 4-bit (frozen).
2. Attach small trainable LoRA matrices in higher precision (usually BF16/FP16).
3. During the forward pass the 4-bit weights are dequantised on the fly, the LoRA path is added, and gradients flow only through the adapters.

Result: you can fine-tune 7 B–13 B models on a single 16–24 GB GPU.

## 3. BitsAndBytesConfig

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4
    bnb_4bit_compute_dtype=torch.float16,  # compute in fp16/bf16
    bnb_4bit_use_double_quant=True,     # nested quantisation for extra savings
)

print(bnb_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



## 4. Load a Small Causal LM in 4-bit

We use a very small model so the notebook runs everywhere.  
Replace with `"TinyLlama/TinyLlama-1.1B-Chat-v1.0"`, `"microsoft/phi-2"`, or a 7 B model when you have enough VRAM.

In [8]:
# model_name = "sshleifer/tiny-gpt2"  # tiny placeholder – swap for a real small LM

# With one of these (choose according to your hardware)
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"     # Best for learning + still small
# model_name = "microsoft/phi-2"                     # Stronger, needs more memory
# model_name = "gpt2"                                # Classic small GPT-2 (124M)

# For a more realistic (but still modest) run try:
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
else:
    # CPU fallback – full precision, no quantisation
    model = AutoModelForCausalLM.from_pretrained(model_name)
    print("Running in full precision on CPU (4-bit requires CUDA).")

print(model)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Running in full precision on CPU (4-bit requires CUDA).
LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
   

## 5. Attach LoRA Adapters (QLoRA)

Run this code if you get any version errors
```
# Upgrade torchao to a compatible version
!pip install -U "torchao>=0.16.0" --quiet

# (Optional but recommended) also make sure peft is recent
!pip install -U peft --quiet
```

In [9]:
# Upgrade torchao to a compatible version
!pip install -U "torchao>=0.16.0" --quiet

# (Optional but recommended) also make sure peft is recent
!pip install -U peft --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 55.9 MB/s eta 0:00:00


In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    # Target modules depend on the architecture.
    # For GPT-2 style: "c_attn"
    # For LLaMA / Mistral / TinyLlama: ["q_proj", "k_proj", "v_proj", "o_proj"]
    # target_modules=["c_attn"] if "gpt2" in model_name.lower() else ["q_proj", "v_proj"],  #correct for other
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # correct for TinyLlama / LLaMA style
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


## 6. Instruction-Tuning Dataset

Instruction tuning teaches the model to follow natural-language instructions.

A typical sample looks like:

```
### Instruction:
Explain what a neural network is.

### Response:
A neural network is a computational model ...
```

We use a tiny public instruction dataset for demonstration. In practice you would use Alpaca, OpenHermes, OpenAssistant, or your own domain data.

In [11]:
# Small instruction-style dataset (replace with your own for real work)
# Using a filtered subset of databricks-dolly-15k for illustration
try:
    raw = load_dataset("databricks/databricks-dolly-15k", split="train")
    raw = raw.shuffle(seed=42).select(range(500))  # tiny slice
except Exception as e:
    print("Could not load dolly – creating a synthetic toy set.")
    print(e)
    from datasets import Dataset
    toy = {
        "instruction": [
            "What is the capital of France?",
            "Explain gravity in one sentence.",
            "Write a short greeting.",
            "What is 2 + 2?",
            "Name three primary colors.",
        ] * 40,
        "context": [""] * 200,
        "response": [
            "The capital of France is Paris.",
            "Gravity is the force that attracts objects toward each other.",
            "Hello! How can I help you today?",
            "2 + 2 equals 4.",
            "The three primary colors are red, blue, and yellow.",
        ] * 40,
    }
    raw = Dataset.from_dict(toy)

print(raw)
print(raw[0])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 500
})
{'instruction': 'Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?', 'context': '', 'response': 'Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree', 'category': 'open_qa'}


In [12]:
def format_instruction(example):
    """Turn a row into a single training string."""
    instruction = example.get("instruction", "")
    context = example.get("context", "") or example.get("input", "")
    response = example.get("response", "") or example.get("output", "")

    if context and context.strip():
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Context:\n{context}\n\n"
            f"### Response:\n{response}"
        )
    else:
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{response}"
        )
    return {"text": prompt}


dataset = raw.map(format_instruction)
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
print(dataset[0]["text"][:300])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

### Instruction:
Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?

### Response:
Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Hern


## 7. Training with SFTTrainer (TRL)

`SFTTrainer` from the TRL library is the standard high-level helper for supervised fine-tuning of causal LMs. It handles packing, label masking, and the instruction format.

In [13]:
training_args = SFTConfig(
    output_dir="./results-qlora-instruct",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,           # ← Changes from warmup_ratio=0.03
    logging_steps=10,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    optim="paged_adamw_8bit" if device == "cuda" else "adamw_torch",
    report_to="none",
    max_length=256,                 # ← changed from max_seq_length
    dataset_text_field="text",
    packing=False,          # set True for longer datasets / higher throughput
    use_cpu=True,        # added to force CPU
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,   # newer TRL API (replaces tokenizer=)
)

print("Trainer ready.")

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4120 > 2048). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Trainer ready.


In [ ]:
# Short training run – increase epochs / data for real quality
train_result = trainer.train()
print(train_result)

## 8. Generate with the Fine-Tuned Adapter

In [15]:
model.eval()

def ask(instruction, max_new_tokens=80):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    # Return only the response part when possible
    if "### Response:" in full:
        return full.split("### Response:")[-1].strip()
    return full


print(ask("What is the capital of France?"))
print("---")
print(ask("Explain gravity in one sentence."))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The capital of France is Paris.
---
Gravity is the force that pulls objects towards the Earth. It is a natural phenomenon that is experienced on a daily basis. Gravity is the force that keeps planets in orbit around the sun and keeps us on the ground.


## 9. Save the QLoRA Adapter

In [ ]:
adapter_dir = "./my-qlora-instruct-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter saved to {adapter_dir}")

# Later reload:
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
# model = PeftModel.from_pretrained(base, adapter_dir)

## 10. Practical Tips for Real QLoRA Runs

| Tip | Recommendation |
|-----|----------------|
| Model size | 7 B fits on 16–24 GB with QLoRA; 13 B needs ~24 GB or offloading |
| Rank | Start with `r=16` or `r=32` for instruction tuning |
| Learning rate | 1e-4 – 2e-4 is common; use cosine schedule + short warmup |
| Batch size | Gradient accumulation to reach effective batch 16–64 |
| Sequence length | 512–2048 depending on VRAM; enable packing for efficiency |
| Dataset quality | Clean, diverse instructions beat raw size |
| Evaluation | Keep a small held-out instruction set and inspect generations |
| Merging | `merge_and_unload()` after training if you need a single checkpoint |

## 11. Full FT vs LoRA vs QLoRA

| Method | Memory | Trainable params | Typical use |
|--------|--------|------------------|---------------------|
| Full fine-tuning | Very high | 100% | Small models, max quality |
| LoRA | Medium | ~0.1–1% | Medium models, multi-adapter serving |
| **QLoRA** | **Low** | ~0.1–1% | **7 B–70 B on consumer GPUs** |

## 12. Summary

| Concept | Description |
|---------|-------------|
| **NF4** | 4-bit NormalFloat quantisation optimised for weight distributions |
| **Double quantisation** | Quantises the quantisation constants themselves for extra savings |
| **QLoRA** | Frozen 4-bit base model + trainable LoRA adapters in higher precision |
| **Instruction tuning** | Supervised fine-tuning on (instruction, response) pairs |
| **SFTTrainer** | TRL helper that simplifies causal-LM supervised fine-tuning |

### Canonical QLoRA snippet

```python
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
```

---

**You have now completed the `03_finetuning` [series](https://github.com/S33mi/modern-ai-llm-journey/blob/main/03_finetuning/).**

Next folder in the curriculum: **`04_rag_systems`**  
→ [`01_chunking_embeddings_vectorstore.ipynb`](https://github.com/S33mi/modern-ai-llm-journey/blob/main/04_rag_systems/)

---

**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Analytics and ML/AI related opportunities